In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships


In [2]:
# The roster the hybrid assembly ran off, and the ONT assemblies it started from
roster = pd.read_csv(TABLES / "KASPAH_QC_outcome_v2.tsv", sep="\t")
external = pd.read_csv(TABLES / "external_qc_summary.tsv", sep="\t")
internal = pd.read_csv(TABLES / "internal_qc_summary.tsv", sep="\t").set_index("sample")
meta = pd.read_csv(TABLES / "KASPAH_collection_nr_170719.tsv", sep="\t").set_index("sample ID")

expected = roster[roster["assemble"] == True]["sample"]
external = external[~external["external_qc_decision"].str.startswith("DROP")].set_index("sample")

len(expected), len(external)

(294, 294)

In [3]:
# What 08_hybrid_asm produced
HYBRID = "../08_hybrid_asm/{sample}/hybrid_clinopore/{sample}_hybrid.fasta"

built = [sample for sample in expected if Path(HYBRID.format(sample=sample)).exists()]
missing = sorted(set(expected) - set(built))

len(built), missing

(294, [])

In [4]:
# Libraries are named for the isolate, the colony pick and the sequencing round:
# INF255.1_v2 is isolate INF255, morphology 1, resequenced in v2. One library survives per
# isolate, so both names identify the same genome and both are worth carrying.
def isolate_of(library):
    return re.sub(r"\.\d+$", "", re.sub(r"_v\d+$", "", library))


def morphology_of(library):
    match = re.search(r"\.(\d+)(?:_v\d+)?$", library)
    return match.group(1) if match else ""

In [5]:
# --- what the assemblies look like ------------------------------------------------------
CHROMOSOME_MIN_LEN = 4_000_000          # bin/scripts/integrate_qc.py definition
KPSC_CHROMOSOME_SIZE = (4_800_000, 6_200_000)


def contig_lengths(path):
    lengths, current = [], 0
    for line in open(path):
        if line.startswith(">"):
            lengths.append(current)
            current = 0
        else:
            current += len(line.strip())
    return sorted((length for length in lengths[1:] + [current] if length), reverse=True)


def assembly_stats(path, prefix):
    lengths = contig_lengths(path)
    chromosomes = [length for length in lengths if length >= CHROMOSOME_MIN_LEN]
    return {f"{prefix}_contigs": len(lengths),
            f"{prefix}_total_len": sum(lengths),
            f"{prefix}_chromosome_len": lengths[0],
            f"{prefix}_n_chromosomes": len(chromosomes)}

In [6]:
# The ONT assembly each hybrid was polished from. Five came out of 06_asm_fixing rather
# than 03_assembly, and INF205 out of the Autocycler consensus, so those are measured from
# their FASTAs; the rest carry usable asm_* columns from internal QC.
FIXED_ASSEMBLY = {
    "INF205": "../06_asm_fixing/01_autocycler/assemblies/INF205.fasta",
    **{strain: f"../06_asm_fixing/02_decontam/assemblies/{strain}.fasta"
       for strain in ["INF186", "INF216", "KSB1_2I", "KSB2_4A", "KSB2_5D_v2"]},
}


def ont_stats(sample):
    if sample in FIXED_ASSEMBLY:
        return assembly_stats(FIXED_ASSEMBLY[sample], "ont")
    row = internal.loc[sample]
    return {"ont_contigs": int(row["asm_num_seqs"]),
            "ont_total_len": int(row["asm_sum_len"]),
            "ont_chromosome_len": int(row["asm_max_len"]),
            "ont_n_chromosomes": int(row["asm_n_chromosomes"])}


assemblies = pd.DataFrame([
    {"library": sample,
     **assembly_stats(HYBRID.format(sample=sample), "hybrid"),
     **ont_stats(sample)}
    for sample in built]).set_index("library")

assemblies["d_contigs"] = assemblies["hybrid_contigs"] - assemblies["ont_contigs"]
assemblies["d_total_len"] = assemblies["hybrid_total_len"] - assemblies["ont_total_len"]
assemblies["d_chromosome_len"] = (assemblies["hybrid_chromosome_len"]
                                  - assemblies["ont_chromosome_len"])

assemblies[["hybrid_contigs", "hybrid_total_len", "hybrid_chromosome_len",
            "d_chromosome_len"]].describe().loc[["min", "50%", "max"]]

,hybrid_contigs,hybrid_total_len,hybrid_chromosome_len,d_chromosome_len
min,1.0,5095397.0,5031183.0,-16316.0
50%,2.0,5519564.5,5307218.0,10.0
max,16.0,6488464.0,5660135.0,48712.0


In [7]:
# Polishing was structurally neutral almost everywhere: the chromosome moves by a median of
# 10 bp, and three quarters of the cohort sits inside 32 bp.
assemblies["d_chromosome_len"].abs().quantile([.5, .75, .90, .95, .99]).astype(int)

0.50       17
0.75       31
0.90      121
0.95      986
0.99    16973
Name: d_chromosome_len, dtype: int64

In [8]:
# The threshold goes in the gap the first pass exposed - the sorted changes ran 25.7 kb,
# 33.5 kb, 48.7 kb, then nothing until 95.1 kb, 119.1 kb, 405.8 kb, 1.86 Mb, 2.06 Mb.
MAX_CHROMOSOME_DELTA = 50_000

assemblies["has_chromosome"] = (
    (assemblies["hybrid_n_chromosomes"] == 1)
    & assemblies["hybrid_chromosome_len"].between(*KPSC_CHROMOSOME_SIZE))
assemblies["chromosome_preserved"] = (
    assemblies["d_chromosome_len"].abs() <= MAX_CHROMOSOME_DELTA)

pd.crosstab(assemblies["has_chromosome"], assemblies["chromosome_preserved"])

chromosome_preserved,True
has_chromosome,
True,294


In [9]:
# --- typing, from all four sources ------------------------------------------------------
# reference   the historical short-read typing in the collection table
# illumina    the Illumina assembly of this library      (04_verification)
# ont         the ONT assembly the hybrid was polished from
# hybrid      the polished assembly, i.e. the final genome
SPECIES = "enterobacterales__species__species"
ST = "klebsiella_pneumo_complex__mlst__ST"
K_LOCUS = "klebsiella_pneumo_complex__kaptive__K_locus"
K_CONFIDENCE = "klebsiella_pneumo_complex__kaptive__K_locus_confidence"
O_LOCUS = "klebsiella_pneumo_complex__kaptive__O_locus"
LOCI = ["gapA", "infB", "mdh", "pgi", "phoE", "rpoB", "tonB"]

verification = pd.read_csv(
    "../04_verification/kleborate/results/klebsiella_pneumo_complex_output.txt", sep="\t")
decontam = pd.read_csv(TABLES / "decontam_typing_resolved.tsv", sep="\t")
hybrid_typing = pd.read_csv(
    "kleborate/results/klebsiella_pneumo_complex_output.txt", sep="\t").set_index("strain")

illumina_typing = verification[verification["strain"].str.endswith("_illumina")].copy()
illumina_typing["strain"] = illumina_typing["strain"].str.removesuffix("_illumina")
illumina_typing = illumina_typing.set_index("strain")

# the decontaminated strains were re-typed after reassembly, so those rows win
ont_only = verification[verification["strain"].str.endswith("_ont")].copy()
ont_only["strain"] = ont_only["strain"].str.removesuffix("_ont")
ont_typing = pd.concat([ont_only.set_index("strain"), decontam.set_index("strain")])
ont_typing = ont_typing[~ont_typing.index.duplicated(keep="last")]

for name, table in [("illumina", illumina_typing), ("ont", ont_typing),
                    ("hybrid", hybrid_typing)]:
    assert set(built) <= set(table.index), name

In [10]:
typing = pd.DataFrame(index=pd.Index(built, name="library"))
typing["isolate"] = [isolate_of(s) for s in built]

for prefix, table, columns in [
        ("hybrid", hybrid_typing, {SPECIES: "species", ST: "ST", K_LOCUS: "K_locus",
                                   K_CONFIDENCE: "K_locus_confidence", O_LOCUS: "O_locus"}),
        ("ont", ont_typing, {SPECIES: "species", ST: "ST", K_LOCUS: "K_locus"}),
        ("illumina", illumina_typing, {SPECIES: "species", ST: "ST", K_LOCUS: "K_locus"})]:
    for source, name in columns.items():
        typing[f"{prefix}_{name}"] = typing.index.map(table[source])

for source, name in {"species": "species", "ST": "ST", "K_locus": "K_locus"}.items():
    typing[f"reference_{name}"] = typing["isolate"].map(meta[source])

typing.head(3)

,isolate,hybrid_species,hybrid_ST,hybrid_K_locus,hybrid_K_locus_confidence,hybrid_O_locus,ont_species,ont_ST,ont_K_locus,illumina_species,illumina_ST,illumina_K_locus,reference_species,reference_ST,reference_K_locus
library,,,,,,,,,,,,,,,
INF001.1,INF001,Klebsiella variicola subsp. variicola,ST3090,KL3,Typeable,OL3α/β,Klebsiella variicola subsp. variicola,ST3090-1LV,KL3,Klebsiella variicola subsp. variicola,ST3090,KL3,Klebsiella variicola subsp. variicola,ST3090,KL3
INF002,INF002,Klebsiella pneumoniae,ST133,KL116,Typeable,OL2α.1,Klebsiella pneumoniae,ST133,KL116,Klebsiella pneumoniae,ST133,KL116,Klebsiella pneumoniae,ST133,KL116
INF003,INF003,Klebsiella pneumoniae,ST323,KL21,Typeable,OL3γ,Klebsiella pneumoniae,ST323,KL21,Klebsiella pneumoniae,ST323,KL21,Klebsiella pneumoniae,ST323,KL21


In [11]:
# MLST is compared by allele rather than by ST name, as in 07_external_qc: "ST1234-2LV" is
# a nearest neighbour rather than a call, and an older MLST database can give the same
# seven alleles a different name.
MISSING = {"-", "", "nan", "NaN", "0"}


def allele(value):
    return re.sub(r"[*^?]", "", str(value)).strip()


def allele_distance(left, right):
    return sum(1 for locus in LOCI
               if allele(left[locus]) in MISSING or allele(right[locus]) in MISSING
               or allele(left[locus]) != allele(right[locus]))


def mlst(table, sample):
    row = table.loc[sample]
    return {locus: row["klebsiella_pneumo_complex__mlst__" + locus] for locus in LOCI}


MLST_MAX_DISTANCE = 2

typing["mlst_distance_ont_to_hybrid"] = [
    allele_distance(mlst(hybrid_typing, s), mlst(ont_typing, s)) for s in built]
typing["mlst_distance_illumina_to_hybrid"] = [
    allele_distance(mlst(hybrid_typing, s), mlst(illumina_typing, s)) for s in built]
typing["mlst_distance_reference_to_hybrid"] = [
    allele_distance(mlst(hybrid_typing, s), meta.loc[isolate_of(s), LOCI]) for s in built]

typing[["mlst_distance_ont_to_hybrid", "mlst_distance_illumina_to_hybrid",
        "mlst_distance_reference_to_hybrid"]].apply(pd.Series.value_counts).fillna(0).astype(int)

,mlst_distance_ont_to_hybrid,mlst_distance_illumina_to_hybrid,mlst_distance_reference_to_hybrid
0,294,294,280
1,0,0,3
3,0,0,1
4,0,0,2
5,0,0,3
6,0,0,1
7,0,0,4


In [12]:
# Not one MLST allele changed between the ONT assembly and its hybrid, anywhere.
typing["species_ont_to_hybrid"] = typing["hybrid_species"] == typing["ont_species"]
typing["K_locus_ont_to_hybrid"] = typing["hybrid_K_locus"] == typing["ont_K_locus"]
typing["typing_concordant"] = (typing["species_ont_to_hybrid"]
                               & typing["K_locus_ont_to_hybrid"]
                               & (typing["mlst_distance_ont_to_hybrid"] <= MLST_MAX_DISTANCE))

typing[["species_ont_to_hybrid", "K_locus_ont_to_hybrid", "typing_concordant"]].sum()

species_ont_to_hybrid    294
K_locus_ont_to_hybrid    294
typing_concordant        294
dtype: int64

In [13]:
# The ST *name* does change for seven, and in the right direction: an inexact allele that
# left Kleborate calling a nearest neighbour is corrected by the Illumina reads, so the ST
# resolves to an exact call. All seven land on the ST the reference collection recorded.
improved = typing[typing["hybrid_ST"] != typing["ont_ST"]]
improved[["isolate", "ont_ST", "hybrid_ST", "reference_ST",
          "mlst_distance_ont_to_hybrid"]]

,isolate,ont_ST,hybrid_ST,reference_ST,mlst_distance_ont_to_hybrid
library,,,,,
INF001.1,INF001,ST3090-1LV,ST3090,ST3090,0
INF007,INF007,ST1449-1LV,ST1449,ST1449,0
INF072,INF072,ST133-1LV,ST133,ST133,0
INF155,INF155,ST882-1LV,ST882,ST882,0
INF200,INF200,ST292-1LV,ST292,ST292,0
INF360,INF360,ST323-1LV,ST323,ST323,0
KSB1_1D,KSB1_1D,ST45-1LV,ST45,ST45,0


In [14]:
assert (improved["hybrid_ST"] == improved["reference_ST"]).all()

In [15]:
# --- decisions --------------------------------------------------------------------------
# A hybrid is kept when it has one chromosome of a credible size, that chromosome is the
# one the ONT assembly had, and polishing did not change what the genome types as.
qc = assemblies.join(typing)


def decide(row):
    if not row["typing_concordant"]:
        return "FAIL_TYPING_CHANGED"
    if not row["has_chromosome"]:
        return "FAIL_CHROMOSOME_SPLIT"
    if not row["chromosome_preserved"]:
        return "FAIL_CHROMOSOME_CHANGED"
    return "PASS"


qc["final_qc_decision"] = qc.apply(decide, axis=1)

# anything that never assembled would land here too
absent = pd.DataFrame({"final_qc_decision": "FAIL_NO_ASSEMBLY"},
                      index=pd.Index(missing, name="library"))
qc = pd.concat([qc, absent])
qc["isolate"] = qc.index.map(isolate_of)

qc["final_qc_decision"].value_counts()

final_qc_decision
PASS    294
Name: count, dtype: int64

In [16]:
# Carry the verdict from every earlier stage, so one row tells the whole history of a genome
qc["internal_qc_decision"] = qc.index.map(internal["internal_qc_decision"])
qc["external_qc_decision"] = qc.index.map(external["external_qc_decision"])
qc["external_mismatch_reason"] = qc.index.map(external["mismatch_reason"]).fillna("")
qc["matches_reference"] = qc.index.map(external["matches_reference"])
qc["morphology"] = qc.index.map(morphology_of)
qc["resequenced"] = qc.index.str.contains(r"_v\d+$", regex=True)

# Concatenating the (possibly empty) no-assembly frame upcasts every integer column to
# float, which would put "5031183.0" in a results table. Nullable Int64 keeps them whole
# and still allows a blank for a library that produced no assembly.
INTEGER = ["hybrid_contigs", "hybrid_total_len", "hybrid_chromosome_len",
           "hybrid_n_chromosomes", "ont_contigs", "ont_total_len", "ont_chromosome_len",
           "d_contigs", "d_total_len", "d_chromosome_len",
           "mlst_distance_ont_to_hybrid", "mlst_distance_illumina_to_hybrid",
           "mlst_distance_reference_to_hybrid"]
qc[INTEGER] = qc[INTEGER].astype("Int64")

qc[["internal_qc_decision", "external_qc_decision", "final_qc_decision"]].value_counts()

internal_qc_decision             external_qc_decision     final_qc_decision
PASS                             PASS                     PASS                 265
                                 PASS_REFERENCE_MISMATCH  PASS                  15
PASS_PROVISIONAL_CHECK_EXTERNAL  PASS                     PASS                   4
PASS_BORDERLINE_MAPPING          PASS                     PASS                   3
PASS_FALSE_POSITIVE_CONTAM       PASS                     PASS                   2
REDO_DECONTAM_KLEBSIELLA         PASS                     PASS                   2
REDO_DECONTAM_ECOLI              PASS                     PASS                   2
REDO_ASSEMBLY_META               PASS                     PASS                   1
Name: count, dtype: int64

In [17]:
# --- how each genome was actually made ---------------------------------------------------
# Four routes, and every genome took one of them. Read from the roster and the repolish
# targets rather than listed here, so the column cannot drift from what was run.
#
#   clinopore                      Flye -> Medaka -> Polypolish -> POLCA, all of it
#                                  clinopore-nf, from the reads in 02_reads
#   kraken2 decontam > clinopore   same, but the ONT reads had a second organism removed
#                                  first (06_asm_fixing/02_decontam)
#   flye (03_assembly) > repolish  clinopore-nf's Flye stage failed or broke a chromosome,
#                                  so the polishing was run over the ONT assembly that
#                                  already existed (08_hybrid_asm/repolish.sh)
#   autocycler > repolish          same, starting from the multi-assembler consensus that
#                                  06_asm_fixing built because Flye cannot resolve INF205
repolish_targets = pd.read_csv(
    TABLES / "repolish_targets.tsv", sep="	", comment="#", dtype=str,
    names=["library", "source_assembly"]).dropna(subset=["library"]).set_index("library")

read_prep = roster.set_index("sample")["read_prep"]


def assembly_route(library):
    if library in repolish_targets.index:
        source = repolish_targets.loc[library, "source_assembly"]
        origin = "autocycler" if "01_autocycler" in source else "flye (03_assembly)"
        return f"{origin} > repolish"
    if read_prep.get(library) == "kraken2_decontam":
        return "kraken2 decontam > clinopore"
    return "clinopore"


qc["read_prep"] = qc.index.map(read_prep)
qc["assembly_route"] = qc.index.map(assembly_route)

qc.groupby(["assembly_route", "read_prep"]).size()

assembly_route                 read_prep       
autocycler > repolish          standard              1
clinopore                      standard            283
flye (03_assembly) > repolish  standard              6
kraken2 decontam > clinopore   kraken2_decontam      4
dtype: int64

In [18]:
# What to do about a row that is not a PASS. Empty for every genome as things stand, and
# kept so a future re-run says what to try rather than only that something is wrong.
#
# All seven failures on the first pass were the same problem - clinopore-nf's Flye stage -
# and all seven were fixed the same way, by running the rest of the pipeline over the ONT
# assembly that already existed. That includes INF074 and KSB2_3C, where Flye produced
# nothing at all: depth-capping or a newer Flye would have been the other route, but
# repolishing what 03_assembly had already built turned out to be enough.
REPOLISH = "add to ../pipeline_tables/repolish_targets.tsv and run repolish.sh"

REMEDY = {
    "PASS": "",
    "FAIL_NO_ASSEMBLY": REPOLISH,
    "FAIL_CHROMOSOME_SPLIT": REPOLISH,
    "FAIL_CHROMOSOME_CHANGED": REPOLISH,
    # never seen: polishing changing species, K locus or an MLST allele would mean the two
    # read sets are not the same genome, which is a provenance problem, not an assembly one
    "FAIL_TYPING_CHANGED": "ONT and Illumina may be different genomes - investigate before use",
}
qc["remedy"] = qc["final_qc_decision"].map(REMEDY)

qc.groupby(["final_qc_decision", "remedy"]).size()

final_qc_decision  remedy
PASS                         294
dtype: int64

In [19]:
COLUMNS = [
    # names: the isolate this genome represents, and the library it was sequenced as
    "isolate", "library", "morphology", "resequenced",
    # verdict at each stage
    "internal_qc_decision", "external_qc_decision", "external_mismatch_reason",
    "final_qc_decision", "remedy",
    # how this genome was produced
    "read_prep", "assembly_route",
    # the final genome
    "hybrid_species", "hybrid_ST", "hybrid_K_locus", "hybrid_K_locus_confidence",
    "hybrid_O_locus",
    # and what the other three sources say it is
    "reference_species", "reference_ST", "reference_K_locus",
    "ont_species", "ont_ST", "ont_K_locus",
    "illumina_species", "illumina_ST", "illumina_K_locus",
    # agreement
    "matches_reference", "species_ont_to_hybrid", "K_locus_ont_to_hybrid",
    "mlst_distance_ont_to_hybrid", "mlst_distance_illumina_to_hybrid",
    "mlst_distance_reference_to_hybrid", "typing_concordant",
    # assembly
    "hybrid_contigs", "hybrid_total_len", "hybrid_chromosome_len", "hybrid_n_chromosomes",
    "ont_contigs", "ont_total_len", "ont_chromosome_len",
    "d_contigs", "d_total_len", "d_chromosome_len",
    "has_chromosome", "chromosome_preserved",
]

summary = qc.reset_index().rename(columns={"index": "library"})[COLUMNS].sort_values("isolate")
summary.to_csv("final_qc_summary.tsv", sep="\t", index=False)
summary.head()

,isolate,library,morphology,resequenced,internal_qc_decision,external_qc_decision,external_mismatch_reason,final_qc_decision,remedy,read_prep,assembly_route,hybrid_species,hybrid_ST,hybrid_K_locus,hybrid_K_locus_confidence,hybrid_O_locus,reference_species,reference_ST,reference_K_locus,ont_species,ont_ST,ont_K_locus,illumina_species,illumina_ST,illumina_K_locus,matches_reference,species_ont_to_hybrid,K_locus_ont_to_hybrid,mlst_distance_ont_to_hybrid,mlst_distance_illumina_to_hybrid,mlst_distance_reference_to_hybrid,typing_concordant,hybrid_contigs,hybrid_total_len,hybrid_chromosome_len,hybrid_n_chromosomes,ont_contigs,ont_total_len,ont_chromosome_len,d_contigs,d_total_len,d_chromosome_len,has_chromosome,chromosome_preserved
0,INF001,INF001.1,1,False,PASS,PASS,,PASS,,standard,clinopore,Klebsiella variicola subsp. variicola,ST3090,KL3,Typeable,OL3α/β,Klebsiella variicola subsp. variicola,ST3090,KL3,Klebsiella variicola subsp. variicola,ST3090-1LV,KL3,Klebsiella variicola subsp. variicola,ST3090,KL3,True,True,True,0,0,0,True,2,5658607,5571684,1,2,5658601,5571678,0,6,6,True,True
1,INF002,INF002,,False,PASS,PASS,,PASS,,standard,clinopore,Klebsiella pneumoniae,ST133,KL116,Typeable,OL2α.1,Klebsiella pneumoniae,ST133,KL116,Klebsiella pneumoniae,ST133,KL116,Klebsiella pneumoniae,ST133,KL116,True,True,True,0,0,0,True,3,5627296,5425360,1,4,5733472,5425252,-1,-106176,108,True,True
2,INF003,INF003,,False,PASS,PASS,,PASS,,standard,clinopore,Klebsiella pneumoniae,ST323,KL21,Typeable,OL3γ,Klebsiella pneumoniae,ST323,KL21,Klebsiella pneumoniae,ST323,KL21,Klebsiella pneumoniae,ST323,KL21,True,True,True,0,0,0,True,3,5534757,5232791,1,3,5534700,5232856,0,57,-65,True,True
3,INF004,INF004,,False,PASS,PASS,,PASS,,standard,clinopore,Klebsiella pneumoniae,ST345,KL15,Typeable,OL4,Klebsiella pneumoniae,ST345,KL15,Klebsiella pneumoniae,ST345,KL15,Klebsiella pneumoniae,ST345,KL15,True,True,True,0,0,0,True,3,5366531,5266612,1,3,5359970,5266566,0,6561,46,True,True
4,INF005,INF005,,False,PASS,PASS,,PASS,,standard,clinopore,Klebsiella variicola subsp. variicola,ST641,KL60,Typeable,OL5,Klebsiella variicola subsp. variicola,ST641,KL60,Klebsiella variicola subsp. variicola,ST641,KL60,Klebsiella variicola subsp. variicola,ST641,KL60,True,True,True,0,0,0,True,8,5985226,5570501,1,8,5985259,5570473,0,-33,28,True,True


In [20]:
# --- sanity check -----------------------------------------------------------------------
kept = summary[summary["final_qc_decision"] == "PASS"]

# one chromosome each, of a credible size, with the identity of the assembly it came from
assert (kept["hybrid_n_chromosomes"] == 1).all()
assert kept["hybrid_chromosome_len"].between(*KPSC_CHROMOSOME_SIZE).all()
assert kept["typing_concordant"].all()
assert (kept["d_chromosome_len"].abs() <= MAX_CHROMOSOME_DELTA).all()

# one row per library, one library per isolate, and the roster fully accounted for
assert not summary["library"].duplicated().any()
assert not kept["isolate"].duplicated().any()
assert set(summary["library"]) == set(expected)

len(kept), kept["isolate"].nunique()

(294, 294)

In [21]:
# Every genome in 10_final_assemblies, and where its identity is agreed
print(f"{len(kept)} assemblies, {kept['isolate'].nunique()} isolates")
print(f"  chromosome        {kept['hybrid_chromosome_len'].min():,} - "
      f"{kept['hybrid_chromosome_len'].max():,} bp")
print(f"  contigs           {kept['hybrid_contigs'].min()} - {kept['hybrid_contigs'].max()}"
      f" (total {kept['hybrid_contigs'].sum()})")
print(f"  agree with the reference collection  {int(kept['matches_reference'].sum())}")
print(f"  sound, reference row disagrees       {int((~kept['matches_reference']).sum())}")

294 assemblies, 294 isolates
  chromosome        5,031,183 - 5,660,135 bp
  contigs           1 - 16 (total 837)
  agree with the reference collection  279
  sound, reference row disagrees       15
